## Embed Name + Genres + Description $\rightarrow$ Genres, Tags to Chromadb

**Warning**: if you really want to run this and wait for about **250 minutes** then you should run this file. Otherwise, you should have a ChromaDB folder `chroma-data` that contains all data

### Import Necessary Lib

In [2]:
import pandas as pd
from utils.preprocess.imputation.genres_tags.schemas import InsertRequest, RetrieveRequest
from utils.preprocess.imputation.genres_tags.func import insert_tag_batch, retrieve_tag_list

### Load Dataset

In [3]:
DATA_PATH = 'data/output/SteamGames_Clean_1.csv'
df = pd.read_csv(DATA_PATH)

### Extract Necessary Features Only

In [4]:
df = df[["Name", "Type", "Genres", "Description", "Tags", "Rank"]]
df_no_dup = df.drop_duplicates()
# We will use these features to predict tags
df_imputated = df_no_dup.dropna(subset=["Name", "Type", "Genres", 'Description'])

### Split to training, predict
Keep only rows have tag to put into vectorDB

In [5]:
df_training = df_imputated.dropna(subset=["Tags"])
df_training = df_training.reset_index(drop=True)

### Put Genres and Tags to ChromaDB
**You should not** run the cell below because it will take a long time to do it. You should have a prerun, already inserted version of **chromadb** storage( I will send later)

In [6]:
# Only run this once to put tags into ChromaDB, no need for second run
# But if you are break enough just run it anyway:)))
items = []
batch_size = 1000
batch_index = 0        # continue from this batch

for index, row in df_training.iterrows():
    if index < batch_index * batch_size:
        continue
    
    item = InsertRequest(
        name=row['Name'],
        genres=row['Genres'].split(','),
        description=row['Description'],
        rank=index + 1,
        tag_list=row['Tags'].split(',')
    )

    items.append(item)

    if len(items) % batch_size == 0 and len(items) > 0:
        print(f"Insert batch {index // 1000}")
        insert_tag_batch(items)
        print("Insert complete")
        items.clear()

Insert batch 0
Insert complete
Insert batch 1
Insert complete
Insert batch 2
Insert complete
Insert batch 3
Insert complete
Insert batch 4
Insert complete
Insert batch 5
Insert complete
Insert batch 6
Insert complete
Insert batch 7
Insert complete
Insert batch 8
Insert complete
Insert batch 9
Insert complete
Insert batch 10
Insert complete
Insert batch 11
Insert complete
Insert batch 12
Insert complete
Insert batch 13
Insert complete
Insert batch 14
Insert complete
Insert batch 15
Insert complete
Insert batch 16
Insert complete
Insert batch 17
Insert complete
Insert batch 18
Insert complete
Insert batch 19
Insert complete
Insert batch 20
Insert complete
Insert batch 21
Insert complete
Insert batch 22
Insert complete
Insert batch 23
Insert complete
Insert batch 24
Insert complete
Insert batch 25
Insert complete
Insert batch 26
Insert complete


After inserting data to vectorDB, you can run benchmark in `benchmark/tags_retrieval.ipynb` to see how well it performce.